In [1]:
from pathlib import Path
import copy
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights,
    swin_t,
    Swin_T_Weights,
    resnet50,
    ResNet50_Weights,
    efficientnet_b0,
    EfficientNet_B0_Weights,
)

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Confirming CUDA is being used

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", GPU_NAME)

PyTorch: 2.14.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 2. Reproducibility and project paths

In [3]:
SEED = 42

def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "separate_attribute_benchmark"
REPORT_DIR = PROJECT_ROOT / "reports" / "separate_attribute_benchmark"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development


## 3. Loading the CompCars CSV

In [4]:
CSV_PATH = DATA_DIR / "processed" / "compcars_modeling_ready.csv"

if not CSV_PATH.exists():
    candidates = list(DATA_DIR.rglob("compcars_modeling_ready.csv"))

    if len(candidates) == 1:
        CSV_PATH = candidates[0]
    elif len(candidates) > 1:
        print("Multiple CSV candidates found:")
        for candidate in candidates:
            print("-", candidate)
        raise RuntimeError("Set CSV_PATH manually.")
    else:
        raise FileNotFoundError(
            "Could not find compcars_modeling_ready.csv. "
            "Set CSV_PATH to its location."
        )

df = pd.read_csv(CSV_PATH)

print("CSV:", CSV_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())

CSV: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\data\processed\compcars_modeling_ready.csv
Rows: 135419
Columns: ['file_path', 'make', 'model_name', 'body_type', 'width', 'height', 'source_dataset', 'image_hash']


,file_path,make,model_name,body_type,width,height,source_dataset,image_hash
0,/content/drive/MyDrive/capstone-vehicle-metada...,Peugeot,Peugeot SR1,NaN,908,626,compcars,25fa9e56ca4f5bb9e73ffe779a101a98
1,/content/drive/MyDrive/capstone-vehicle-metada...,Peugeot,Peugeot SR1,NaN,908,626,compcars,8558bc2cb0192cc1e11465b874225d58
2,/content/drive/MyDrive/capstone-vehicle-metada...,Peugeot,Peugeot SR1,NaN,800,567,compcars,ac30b864ce7e76e4af2b230754b85ec1
3,/content/drive/MyDrive/capstone-vehicle-metada...,Peugeot,Peugeot SR1,NaN,800,567,compcars,efb05d59635584496daac973569dd50e
4,/content/drive/MyDrive/capstone-vehicle-metada...,Peugeot,Peugeot SR1,NaN,800,567,compcars,62e2ef440fb1a950464ad160c16169b8


## 4. Configuring CompCars image root

In [6]:
DEFAULT_COMPCARS = (
    r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
    r"\capstone-vehicle-metadata\data\compcars_kaggle\extracted"
)

COMPCARS_ROOT = Path(
    os.environ.get(
        "COMPCARS_EXTRACTED",
        DEFAULT_COMPCARS
    )
)

print("CompCars root:", COMPCARS_ROOT)

def resolve_local_path(original_path):
    text = str(original_path).replace("\\", "/")

    if "/extracted/" in text:
        relative = text.split("/extracted/", 1)[1]
    elif "/image/" in text:
        relative = "image/" + text.split("/image/", 1)[1]
    else:
        return Path(original_path)

    return COMPCARS_ROOT / Path(*relative.split("/"))

CompCars root: G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\compcars_kaggle\extracted


## 5. Architectures

In [7]:
TASKS_TO_RUN = [
    "make",
    "make_model",
    "body_type",
]

MODELS_TO_RUN = [
    "convnext_tiny",
    "swin_t",
    "resnet50",
    "efficientnet_b0",
]

IMAGES_PER_CLASS = 100
MIN_IMAGES_PER_CLASS = 100

MAX_CLASSES = {
    "make": 20,
    "make_model": 20,
    "body_type": None,
}

BATCH_SIZE = 16
NUM_WORKERS = 0
IMAGE_SIZE = 224

print("Tasks:", TASKS_TO_RUN)
print("Models:", MODELS_TO_RUN)

Tasks: ['make', 'make_model', 'body_type']
Models: ['convnext_tiny', 'swin_t', 'resnet50', 'efficientnet_b0']


## 6. Building task labels

In [8]:
working_df = df.copy()

working_df["make"] = (
    working_df["make"]
    .astype("string")
    .str.strip()
)

working_df["model_name"] = (
    working_df["model_name"]
    .astype("string")
    .str.strip()
)

working_df["make_model"] = (
    working_df["make"].fillna("")
    + " | "
    + working_df["model_name"].fillna("")
)

BODY_TYPE_COLUMN = (
    "body_type_standardised"
    if "body_type_standardised" in working_df.columns
    else "body_type"
)

print("Body type column:", BODY_TYPE_COLUMN)

if BODY_TYPE_COLUMN in working_df.columns:
    print("\nBody-type counts:")
    display(
        working_df[BODY_TYPE_COLUMN]
        .value_counts(dropna=False)
        .to_frame("count")
    )

Body type column: body_type

Body-type counts:


,count
body_type,
NaN,37901
sedan,32232
SUV,21702
hatchback,16534
MPV,6149
sports,5987
minibus,4638
hardtop convertible,2473
fastback,2473


## 7. Duplicate audit

In [9]:
if "image_hash" in working_df.columns:
    duplicate_rows = working_df["image_hash"].duplicated(
        keep=False
    ).sum()

    duplicate_hashes = working_df.loc[
        working_df["image_hash"].duplicated(keep=False),
        "image_hash",
    ].nunique()

    print("Rows involved in duplicate hashes:", duplicate_rows)
    print("Duplicate hash groups:", duplicate_hashes)

    before = len(working_df)

    working_df = (
        working_df
        .drop_duplicates(
            subset="image_hash",
            keep="first",
        )
        .reset_index(drop=True)
    )

    print("Removed exact duplicate rows:", before - len(working_df))
    print("Rows remaining:", len(working_df))

else:
    print(
        "No image_hash column found. "
        "Exact duplicate removal was skipped."
    )

Rows involved in duplicate hashes: 0
Duplicate hash groups: 0
Removed exact duplicate rows: 0
Rows remaining: 135419


## 8. Creating balanced task-specific datasets and splits

In [10]:
def prepare_task_dataframe(task_name):
    if task_name == "body_type":
        label_column = BODY_TYPE_COLUMN
    else:
        label_column = task_name

    if label_column not in working_df.columns:
        raise KeyError(
            f"Missing label column: {label_column}"
        )

    task_df = working_df.copy()

    task_df["label"] = (
        task_df[label_column]
        .astype("string")
        .str.strip()
    )

    task_df = task_df[
        task_df["label"].notna()
        & (task_df["label"] != "")
        & (task_df["label"].str.lower() != "nan")
        & (task_df["label"].str.lower() != "<na>")
    ].copy()

    counts = task_df["label"].value_counts()

    eligible_classes = counts[
        counts >= MIN_IMAGES_PER_CLASS
    ].index.tolist()

    if MAX_CLASSES[task_name] is not None:
        eligible_classes = (
            counts.loc[eligible_classes]
            .sort_values(ascending=False)
            .head(MAX_CLASSES[task_name])
            .index
            .tolist()
        )

    task_df = task_df[
        task_df["label"].isin(eligible_classes)
    ].copy()

    sampled_parts = []

    for label in eligible_classes:
        class_rows = task_df[
            task_df["label"] == label
        ]

        n = min(
            IMAGES_PER_CLASS,
            len(class_rows),
        )

        sampled_parts.append(
            class_rows.sample(
                n=n,
                random_state=SEED,
            )
        )

    task_df = pd.concat(
        sampled_parts,
        ignore_index=True,
    )

    task_df["local_file_path"] = (
        task_df["file_path"]
        .map(resolve_local_path)
    )

    existing_mask = task_df[
        "local_file_path"
    ].map(Path.exists)

    if not existing_mask.all():
        missing = task_df.loc[
            ~existing_mask,
            ["file_path", "local_file_path"],
        ]

        print(
            f"WARNING: {len(missing)} missing files "
            f"for task {task_name}"
        )

        display(
            missing.head(10)
        )

        task_df = task_df.loc[
            existing_mask
        ].copy()

    labels = sorted(
        task_df["label"].unique()
    )

    label_to_target = {
        label: idx
        for idx, label in enumerate(labels)
    }

    task_df["target"] = (
        task_df["label"]
        .map(label_to_target)
        .astype(int)
    )

    train_df, temp_df = train_test_split(
        task_df,
        test_size=0.20,
        random_state=SEED,
        stratify=task_df["target"],
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_df["target"],
    )

    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    split_df = pd.concat(
        [train_df, val_df, test_df],
        ignore_index=True,
    )

    return split_df, labels


TASK_DATA = {}

for task_name in TASKS_TO_RUN:
    split_df, labels = prepare_task_dataframe(
        task_name
    )

    TASK_DATA[task_name] = {
        "df": split_df,
        "classes": labels,
    }

    print("\n" + "=" * 70)
    print("TASK:", task_name)
    print("Classes:", len(labels))
    print("Rows:", len(split_df))
    print(
        split_df["split"]
        .value_counts()
        .to_dict()
    )

    display(
        split_df.groupby(
            ["label", "split"]
        ).size().unstack(
            fill_value=0
        )
    )

    split_df.to_csv(
        REPORT_DIR
        / f"{task_name}_split.csv",
        index=False,
    )


TASK: make
Classes: 20
Rows: 2000
{'train': 1600, 'val': 200, 'test': 200}


split,test,train,val
label,,,
Audi,10,80,10
BWM,10,80,10
BYD,10,80,10
Benz,10,80,10
Chevy,10,80,10
Chrey,10,80,10
Citroen,10,80,10
Ford,10,80,10
Geely,10,80,10



TASK: make_model
Classes: 20
Rows: 2000
{'train': 1600, 'val': 200, 'test': 200}


split,test,train,val
label,,,
Audi | Audi A5 coupe,10,80,10
Audi | Audi A5 hatchback,10,80,10
Audi | Audi A8L,10,80,10
BAW | BAW E Series hatchback,10,80,10
BWM | BWM 3 Series,10,80,10
BWM | BWM 5 Series,10,80,10
Benz | Benz E Class,10,80,10
Benz | Benz E Class couple,10,80,10
Buck | EXCELLE GT,10,80,10



TASK: body_type
Classes: 12
Rows: 1200
{'train': 960, 'val': 120, 'test': 120}


split,test,train,val
label,,,
MPV,10,80,10
SUV,10,80,10
convertible,10,80,10
crossover,10,80,10
estate,10,80,10
fastback,10,80,10
hardtop convertible,10,80,10
hatchback,10,80,10
minibus,10,80,10


## 9. Leakage sanity check across task splits

In [11]:
for task_name, task_info in TASK_DATA.items():
    split_df = task_info["df"]

    print("\nTASK:", task_name)

    if "image_hash" in split_df.columns:
        split_hashes = {
            split: set(
                split_df.loc[
                    split_df["split"] == split,
                    "image_hash",
                ]
            )
            for split in [
                "train",
                "val",
                "test",
            ]
        }

        print(
            "Train ∩ Val:",
            len(
                split_hashes["train"]
                & split_hashes["val"]
            ),
        )

        print(
            "Train ∩ Test:",
            len(
                split_hashes["train"]
                & split_hashes["test"]
            ),
        )

        print(
            "Val ∩ Test:",
            len(
                split_hashes["val"]
                & split_hashes["test"]
            ),
        )
    else:
        print("image_hash unavailable.")


TASK: make
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

TASK: make_model
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

TASK: body_type
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


## 10. Image transformations and Dataset

In [12]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.80, 1.0),
        ratio=(0.85, 1.15),
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        MEAN,
        STD,
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        MEAN,
        STD,
    ),
])

class VehicleAttributeDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image = Image.open(
            row["local_file_path"]
        ).convert("RGB")

        return (
            self.transform(image),
            int(row["target"]),
        )

def make_loaders(split_df):
    train_df = split_df[
        split_df["split"] == "train"
    ]

    val_df = split_df[
        split_df["split"] == "val"
    ]

    test_df = split_df[
        split_df["split"] == "test"
    ]

    train_loader = DataLoader(
        VehicleAttributeDataset(
            train_df,
            train_transform,
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    val_loader = DataLoader(
        VehicleAttributeDataset(
            val_df,
            eval_transform,
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    test_loader = DataLoader(
        VehicleAttributeDataset(
            test_df,
            eval_transform,
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
    )

## 11. Best architecture configurations from previous experiments

In [13]:
MODEL_CONFIGS = {
    "convnext_tiny": {
        "finetune_lr": 5e-5,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "stage2_epochs": 5,
    },

    "swin_t": {
        "finetune_lr": 3e-5,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "stage2_epochs": 6,
    },

    "resnet50": {
        "finetune_lr": 5e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "stage2_epochs": 3,
    },

    "efficientnet_b0": {
        "finetune_lr": 1e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "stage2_epochs": 7,
    },
}

HEAD_LR = 1e-3
HEAD_EPOCHS = 3

## 12. Model builders and unfreezing strategies

In [14]:
def build_model(
    model_name,
    num_classes,
):
    if model_name == "convnext_tiny":
        model = convnext_tiny(
            weights=ConvNeXt_Tiny_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        model.classifier[2] = nn.Linear(
            model.classifier[2].in_features,
            num_classes,
        )

        head_parameters = (
            model.classifier[2].parameters()
        )

    elif model_name == "swin_t":
        model = swin_t(
            weights=Swin_T_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        model.head = nn.Linear(
            model.head.in_features,
            num_classes,
        )

        head_parameters = (
            model.head.parameters()
        )

    elif model_name == "resnet50":
        model = resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes,
        )

        head_parameters = (
            model.fc.parameters()
        )

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes,
        )

        head_parameters = (
            model.classifier[1].parameters()
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return (
        model.to(DEVICE),
        head_parameters,
    )


def unfreeze_for_stage2(
    model,
    model_name,
):
    if model_name == "convnext_tiny":
        modules = [
            model.features[5],
            model.features[6],
            model.features[7],
            model.classifier[2],
        ]

    elif model_name == "swin_t":
        modules = [
            model.features[5],
            model.features[6],
            model.features[7],
            model.norm,
            model.head,
        ]

    elif model_name == "resnet50":
        modules = [
            model.layer4,
            model.fc,
        ]

    elif model_name == "efficientnet_b0":
        modules = [
            model.features[6],
            model.features[7],
            model.features[8],
            model.classifier[1],
        ]

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    for module in modules:
        for p in module.parameters():
            p.requires_grad = True


def count_trainable(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

## 13. Training and evaluation utilities

In [15]:
def topk_correct(
    logits,
    targets,
    k=5,
):
    k = min(
        k,
        logits.shape[1],
    )

    return (
        logits.topk(
            k,
            dim=1,
        )
        .indices
        .eq(
            targets.view(-1, 1)
        )
        .any(dim=1)
        .sum()
        .item()
    )


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
):
    training = optimizer is not None
    model.train(training)

    loss_sum = 0.0
    top1_correct = 0
    top5_correct = 0
    total = 0

    for images, targets in tqdm(
        loader,
        leave=False,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(images)
            loss = criterion(
                logits,
                targets,
            )

        if training:
            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        batch_size = images.size(0)

        loss_sum += (
            loss.item()
            * batch_size
        )

        top1_correct += (
            logits.argmax(dim=1)
            == targets
        ).sum().item()

        top5_correct += topk_correct(
            logits,
            targets,
            5,
        )

        total += batch_size

    return {
        "loss": loss_sum / total,
        "top1": top1_correct / total,
        "top5": top5_correct / total,
    }


def train_stage(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs,
    stage_name,
):
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )

    history = []
    best_val = -1.0
    best_state = None

    for epoch in range(
        1,
        epochs + 1,
    ):
        start = time.perf_counter()

        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            scaler=scaler,
        )

        with torch.inference_mode():
            val_metrics = run_epoch(
                model,
                val_loader,
                criterion,
            )

        elapsed = (
            time.perf_counter()
            - start
        )

        row = {
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_top1": train_metrics["top1"],
            "train_top5": train_metrics["top5"],
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top5": val_metrics["top5"],
            "seconds": elapsed,
        }

        history.append(row)

        print(
            f"[{stage_name}] "
            f"{epoch}/{epochs} | "
            f"train={row['train_top1']:.3f} "
            f"val={row['val_top1']:.3f} "
            f"top5={row['val_top5']:.3f} | "
            f"{elapsed:.1f}s"
        )

        if row["val_top1"] > best_val:
            best_val = row[
                "val_top1"
            ]

            best_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(
        best_state
    )

    return (
        history,
        best_val,
        best_state,
    )


def evaluate_model(
    model,
    loader,
):
    model.eval()

    y_true = []
    y_pred = []
    probabilities_all = []

    with torch.inference_mode():
        for images, targets in tqdm(
            loader,
            leave=False,
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

            logits = model(images)

            probabilities = torch.softmax(
                logits,
                dim=1,
            )

            y_true.extend(
                targets.numpy()
            )

            y_pred.extend(
                probabilities
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            probabilities_all.append(
                probabilities.cpu()
            )

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    probabilities_all = torch.cat(
        probabilities_all,
        dim=0,
    )

    top1 = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    k = min(
        5,
        probabilities_all.shape[1],
    )

    topk_indices = (
        probabilities_all
        .topk(
            k,
            dim=1,
        )
        .indices
        .numpy()
    )

    top5 = np.mean([
        true_target
        in candidate_targets

        for (
            true_target,
            candidate_targets,
        )
        in zip(
            y_true,
            topk_indices,
        )
    ])

    return {
        "top1": float(top1),
        "top5": float(top5),
        "macro_f1": float(macro_f1),
        "y_true": y_true,
        "y_pred": y_pred,
    }

## 14. Running the benchmark

In [16]:
all_results = []

for task_name in TASKS_TO_RUN:
    task_info = TASK_DATA[
        task_name
    ]

    split_df = task_info[
        "df"
    ]

    class_names = task_info[
        "classes"
    ]

    num_classes = len(
        class_names
    )

    (
        train_loader,
        val_loader,
        test_loader,
    ) = make_loaders(
        split_df
    )

    task_report_dir = (
        REPORT_DIR
        / task_name
    )

    task_model_dir = (
        MODEL_DIR
        / task_name
    )

    task_report_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    task_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for model_name in MODELS_TO_RUN:
        reset_seed()

        config = MODEL_CONFIGS[
            model_name
        ]

        print(
            "\n"
            + "=" * 90
        )

        print(
            "TASK:",
            task_name,
            "| MODEL:",
            model_name,
        )

        print(
            "Classes:",
            num_classes,
        )

        print(
            "Config:",
            config,
        )

        print(
            "=" * 90
        )

        model, head_parameters = build_model(
            model_name,
            num_classes,
        )

        criterion = nn.CrossEntropyLoss(
            label_smoothing=config[
                "label_smoothing"
            ]
        )

        # Stage 1: classifier head
        optimizer = torch.optim.AdamW(
            head_parameters,
            lr=HEAD_LR,
            weight_decay=config[
                "weight_decay"
            ],
        )

        (
            history1,
            val1,
            state1,
        ) = train_stage(
            model,
            train_loader,
            val_loader,
            criterion,
            optimizer,
            HEAD_EPOCHS,
            "head",
        )

        # Stage 2: tuned fine-tuning
        unfreeze_for_stage2(
            model,
            model_name,
        )

        print(
            "Trainable parameters:",
            f"{count_trainable(model):,}",
        )

        optimizer = torch.optim.AdamW(
            filter(
                lambda p: p.requires_grad,
                model.parameters(),
            ),
            lr=config[
                "finetune_lr"
            ],
            weight_decay=config[
                "weight_decay"
            ],
        )

        (
            history2,
            val2,
            state2,
        ) = train_stage(
            model,
            train_loader,
            val_loader,
            criterion,
            optimizer,
            config[
                "stage2_epochs"
            ],
            "fine_tune",
        )

        if val1 > val2:
            best_val = val1
            best_state = state1
            selected_stage = "head"
        else:
            best_val = val2
            best_state = state2
            selected_stage = "fine_tune"

        model.load_state_dict(
            best_state
        )

        # Test evaluation
        test_metrics = evaluate_model(
            model,
            test_loader,
        )

        y_true = test_metrics[
            "y_true"
        ]

        y_pred = test_metrics[
            "y_pred"
        ]

        result_row = {
            "task": task_name,
            "model": model_name,
            "num_classes": num_classes,
            "num_train": int(
                (
                    split_df["split"]
                    == "train"
                ).sum()
            ),
            "num_val": int(
                (
                    split_df["split"]
                    == "val"
                ).sum()
            ),
            "num_test": int(
                (
                    split_df["split"]
                    == "test"
                ).sum()
            ),
            "selected_stage": selected_stage,
            "validation_top1": float(
                best_val
            ),
            "test_top1": test_metrics[
                "top1"
            ],
            "test_top5": test_metrics[
                "top5"
            ],
            "test_macro_f1": test_metrics[
                "macro_f1"
            ],
            "gpu": GPU_NAME,
        }

        all_results.append(
            result_row
        )

        print(
            "\nFINAL:",
            result_row,
        )

        # Saving classification report
        report_df = pd.DataFrame(
            classification_report(
                y_true,
                y_pred,
                labels=list(
                    range(
                        num_classes
                    )
                ),
                target_names=class_names,
                output_dict=True,
                zero_division=0,
            )
        ).T

        report_df.to_csv(
            task_report_dir
            / (
                f"{model_name}"
                "_classification_report.csv"
            )
        )

        # Saving confusion matrix
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=list(
                range(
                    num_classes
                )
            ),
        )

        fig_size = max(
            10,
            min(
                24,
                num_classes * 0.7,
            ),
        )

        fig, ax = plt.subplots(
            figsize=(
                fig_size,
                fig_size,
            )
        )

        ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=class_names,
        ).plot(
            ax=ax,
            xticks_rotation=90,
            colorbar=False,
            values_format="d",
        )

        ax.set_title(
            f"{task_name} — {model_name}"
        )

        plt.tight_layout()

        plt.savefig(
            task_report_dir
            / (
                f"{model_name}"
                "_confusion_matrix.png"
            ),
            dpi=150,
        )

        plt.close(
            fig
        )

        # Saving checkpoint
        torch.save(
            {
                "task": task_name,
                "architecture": model_name,
                "state_dict": best_state,
                "classes": class_names,
                "num_classes": num_classes,
                "config": config,
                "validation_top1": float(
                    best_val
                ),
                "test_top1": test_metrics[
                    "top1"
                ],
                "test_top5": test_metrics[
                    "top5"
                ],
                "test_macro_f1": test_metrics[
                    "macro_f1"
                ],
            },
            task_model_dir
            / f"{model_name}_best.pt",
        )

        del model
        torch.cuda.empty_cache()


TASK: make | MODEL: convnext_tiny
Classes: 20
Config: {'finetune_lr': 5e-05, 'weight_decay': 0.0001, 'label_smoothing': 0.1, 'stage2_epochs': 5}


[head] 1/3 | train=0.096 val=0.140 top5=0.460 | 2122.9s


[head] 2/3 | train=0.234 val=0.175 top5=0.475 | 47.2s


[head] 3/3 | train=0.296 val=0.165 top5=0.500 | 46.8s
Trainable parameters: 26,303,252


[fine_tune] 1/5 | train=0.372 val=0.220 top5=0.585 | 52.2s


...

[fine_tune] 4/7 | train=0.725 val=0.483 top5=0.833 | 37.8s


[fine_tune] 5/7 | train=0.791 val=0.517 top5=0.850 | 36.9s


[fine_tune] 6/7 | train=0.871 val=0.492 top5=0.867 | 37.2s


[fine_tune] 7/7 | train=0.883 val=0.500 top5=0.867 | 36.3s



FINAL: {'task': 'body_type', 'model': 'efficientnet_b0', 'num_classes': 12, 'num_train': 960, 'num_val': 120, 'num_test': 120, 'selected_stage': 'fine_tune', 'validation_top1': 0.5166666666666667, 'test_top1': 0.43333333333333335, 'test_top5': 0.8083333333333333, 'test_macro_f1': 0.4241368882235756, 'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU'}


## 15. Results table

In [17]:
results_df = (
    pd.DataFrame(
        all_results
    )
    .sort_values(
        [
            "task",
            "test_top1",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

display(
    results_df
)

results_df.to_csv(
    REPORT_DIR
    / "all_task_model_results.csv",
    index=False,
)

,task,model,num_classes,num_train,num_val,num_test,selected_stage,validation_top1,test_top1,test_top5,test_macro_f1,gpu
0,body_type,convnext_tiny,12,960,120,120,fine_tune,0.608333,0.583333,0.900000,0.585245,NVIDIA GeForce RTX 5060 Laptop GPU
1,body_type,swin_t,12,960,120,120,fine_tune,0.591667,0.475000,0.883333,0.469391,NVIDIA GeForce RTX 5060 Laptop GPU
2,body_type,efficientnet_b0,12,960,120,120,fine_tune,0.516667,0.433333,0.808333,0.424137,NVIDIA GeForce RTX 5060 Laptop GPU
3,body_type,resnet50,12,960,120,120,fine_tune,0.508333,0.416667,0.816667,0.385563,NVIDIA GeForce RTX 5060 Laptop GPU
4,make,convnext_tiny,20,1600,200,200,fine_tune,0.405000,0.470000,0.800000,0.466903,NVIDIA GeForce RTX 5060 Laptop GPU
5,make,swin_t,20,1600,200,200,fine_tune,0.330000,0.400000,0.730000,0.397024,NVIDIA GeForce RTX 5060 Laptop GPU
6,make,resnet50,20,1600,200,200,fine_tune,0.310000,0.330000,0.695000,0.322522,NVIDIA GeForce RTX 5060 Laptop GPU
7,make,efficientnet_b0,20,1600,200,200,fine_tune,0.260000,0.305000,0.665000,0.286146,NVIDIA GeForce RTX 5060 Laptop GPU
8,make_model,convnext_tiny,20,1600,200,200,fine_tune,0.895000,0.930000,1.000000,0.928928,NVIDIA GeForce RTX 5060 Laptop GPU
9,make_model,resnet50,20,1600,200,200,fine_tune,0.900000,0.905000,0.990000,0.904658,NVIDIA GeForce RTX 5060 Laptop GPU


## 16. Best architecture for each task

In [18]:
best_by_task = (
    results_df
    .sort_values(
        "validation_top1",
        ascending=False,
    )
    .groupby(
        "task",
        as_index=False,
    )
    .first()
)

display(
    best_by_task[
        [
            "task",
            "model",
            "num_classes",
            "validation_top1",
            "test_top1",
            "test_top5",
            "test_macro_f1",
        ]
    ]
)

best_by_task.to_csv(
    REPORT_DIR
    / "best_model_per_task.csv",
    index=False,
)

,task,model,num_classes,validation_top1,test_top1,test_top5,test_macro_f1
0,body_type,convnext_tiny,12,0.608333,0.583333,0.90,0.585245
1,make,convnext_tiny,20,0.405000,0.470000,0.80,0.466903
2,make_model,resnet50,20,0.900000,0.905000,0.99,0.904658
